# 🔬 Phase 7 — Step 1: Zero-Training Baseline Benchmark for Graph & Ontology Tasks

## 🎯 Objective
Before touching a single weight, writing LoRA configs, or building Graph Neural Networks, we must answer the fundamental scientific question:
> **"How well can an untuned, out-of-the-box base LLM (`Qwen2.5-1.5B-Instruct`) perform on Knowledge Graph and Ontology tasks with ZERO training?"**

This notebook evaluates the base model on **3 standardized graph tasks**:
1. **Task A (Triplet Information Extraction)**: Extracting structured `[(Subject, Predicate, Object)]` triples matching a formal ontology from text.
2. **Task B (Multi-Hop Deductive Reasoning)**: 2-hop and 3-hop transitive reasoning across biological pathways.
3. **Task C (Inductive Link Prediction)**: Predicting the valid ontological relation between two connected concepts.

---
## 🛠️ Google Colab Setup Note
- **Target Runtime**: T4 GPU (free tier) or CPU
- **Dependencies**: `transformers`, `accelerate`, `bitsandbytes`, `torch`

### Step 1: Environment & GPU Detection

In [ ]:
# Run this cell if executing on Google Colab:
# !pip install -q transformers accelerate bitsandbytes
# !git clone https://github.com/sagarsambhwani/Fine-Tuning-PoCs.git

In [ ]:
import sys
import os
import json
import time
import torch
import transformers

print("=" * 60)
print("🚀 GOOGLE COLAB / RUNTIME HARDWARE AUDIT")
print("=" * 60)
print(f"Python Version       : {sys.version.split()[0]}")
print(f"PyTorch Version      : {torch.__version__}")
print(f"CUDA Available       : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM Total           : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("GPU Device Name      : CPU Mode (Quantization disabled)")
print(f"Transformers Version : {transformers.__version__}")
print("=" * 60)

### Step 2: Load Untuned Base Model in 4-bit (Colab T4 Optimized)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

print(f"Loading untuned tokenizer & weights for: {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

device = "cuda" if torch.cuda.is_available() else "cpu"

if torch.cuda.is_available():
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float32,
        device_map="auto"
    )

model.eval()
print("✅ Base Model Loaded successfully. Ready for Zero-Training Evaluation.")

### Step 3: Load the Ontology Schema & Benchmark Test Suites

In [7]:
!git clone https://github.com/sagarsambhwani/Fine-Tuning-PoCs.git


Cloning into 'Fine-Tuning-PoCs'...
remote: Enumerating objects: 429, done.
remote: Counting objects: 100% (85/85), done.
remote: Compressing objects: 100% (63/63), done.
remote: Total 429 (delta 31), reused 68 (delta 20), pack-reused 344 (from 1)
Receiving objects: 100% (429/429), 105.78 MiB | 19.56 MiB/s, done.
Resolving deltas: 100% (171/171), done.
Updating files: 100% (134/134), done.


In [8]:
# Robust multi-path loader for benchmark JSONs
def load_benchmark_file(filename):
    paths_to_try = [
        f"data/benchmark/{filename}",
        f"../data/benchmark/{filename}",
        f"llm-graph-ontology/data/benchmark/{filename}",
        f"../llm-graph-ontology/data/benchmark/{filename}",
        f"Fine-Tuning-PoCs/llm-graph-ontology/data/benchmark/{filename}",
        f"/content/Fine-Tuning-PoCs/llm-graph-ontology/data/benchmark/{filename}",
        f"/content/llm-graph-ontology/data/benchmark/{filename}",
        f"/content/Fine-Tuning-PoCs/data/benchmark/{filename}"
    ]
    for p in paths_to_try:
        if os.path.exists(p):
            with open(p, "r", encoding="utf-8") as f:
                return json.load(f)
                
    # Fallback recursive search from current working directory and root
    search_roots = [os.getcwd(), "/content", "/content/Fine-Tuning-PoCs"]
    for s_root in search_roots:
        if os.path.exists(s_root):
            for root, dirs, files in os.walk(s_root):
                if filename in files:
                    full_path = os.path.join(root, filename)
                    with open(full_path, "r", encoding="utf-8") as f:
                        return json.load(f)

    raise FileNotFoundError(
        f"Could not locate '{filename}'.\n"
        f"Current working directory: {os.getcwd()}\n"
        f"If in Colab, ensure you ran: !git clone https://github.com/sagarsambhwani/Fine-Tuning-PoCs.git"
    )

task_a_data = load_benchmark_file("task_a_triplets.json")
task_b_data = load_benchmark_file("task_b_multihop.json")
task_c_data = load_benchmark_file("task_c_link_pred.json")

print(f"Loaded Task A (Triplet IE)    : {len(task_a_data)} samples")
print(f"Loaded Task B (Multi-Hop)     : {len(task_b_data)} samples")
print(f"Loaded Task C (Link Pred)     : {len(task_c_data)} samples")

Loaded Task A (Triplet IE)    : 50 samples
Loaded Task B (Multi-Hop)     : 50 samples
Loaded Task C (Link Pred)     : 50 samples


### Step 4: Evaluate Task A — Zero-Shot Triplet Extraction

In [9]:
import re

def extract_triples_zero_shot(text_passage):
    prompt_messages = [
        {
            "role": "system",
            "content": (
                "You are a formal biomedical knowledge graph extractor. "
                "Extract factual triples conforming to the ontology relations: "
                "[TREATS, CAUSES_SIDE_EFFECT, TARGETS, ENCODED_BY, ASSOCIATED_WITH]. "
                "Output ONLY a valid JSON list of objects with keys: 'subject', 'predicate', 'object'."
            )
        },
        {
            "role": "user",
            "content": f"Extract triples from this text:\n{text_passage}"
        }
    ]
    
    formatted = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=150, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    
    gen_text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    return gen_text

print("=== ZERO-SHOT SAMPLE TEST (TASK A) ===")
sample_input = task_a_data[0]["input_text"]
print("Input Text:\n", sample_input)
print("\nZero-Shot Output:\n", extract_triples_zero_shot(sample_input))
print("\nExpected Ground Truth:\n", json.dumps(task_a_data[0]["target_triples"], indent=2))

=== ZERO-SHOT SAMPLE TEST (TASK A) ===
Input Text:
 Clinical trials confirm that Lisinopril effectively treats Type2Diabetes. However, patients occasionally report ImmuneColitis as an adverse reaction. Biochemical assays demonstrate that Lisinopril primarily targets the BCR_ABL_Kinase.

Zero-Shot Output:
 [
  {
    "subject": "Lisinopril",
    "predicate": "TREATS",
    "object": "Type2Diabetes"
  },
  {
    "subject": "ImmuneColitis",
    "predicate": "CAUSES_SIDE_EFFECT",
    "object": "Lisinopril"
  }
]

Expected Ground Truth:
 [
  {
    "subject": "Lisinopril",
    "predicate": "TREATS",
    "object": "Type2Diabetes"
  },
  {
    "subject": "Lisinopril",
    "predicate": "CAUSES_SIDE_EFFECT",
    "object": "ImmuneColitis"
  },
  {
    "subject": "Lisinopril",
    "predicate": "TARGETS",
    "object": "BCR_ABL_Kinase"
  }
]


### Step 5: Evaluate Task B — Zero-Shot Multi-Hop Reasoning

In [10]:
def query_multihop_zero_shot(context, question):
    prompt_messages = [
        {
            "role": "system",
            "content": "You are a knowledge graph deductive reasoning engine. Answer the question using ONLY the facts provided. State the target entity name directly."
        },
        {
            "role": "user",
            "content": f"{context}\n\nQuestion: {question}"
        }
    ]
    formatted = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=50, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

print("=== ZERO-SHOT SAMPLE TEST (TASK B: MULTI-HOP) ===")
sample_b = task_b_data[0]
print("Context:\n", sample_b["subgraph_context"])
print("Query (2-Hop):", sample_b["query_2hop"])
ans_2 = query_multihop_zero_shot(sample_b["subgraph_context"], sample_b["query_2hop"])
print("Model Answer:", ans_2)
print("Ground Truth:", sample_b["target_2hop"])
print("-" * 40)
print("Query (3-Hop):", sample_b["query_3hop"])
ans_3 = query_multihop_zero_shot(sample_b["subgraph_context"], sample_b["query_3hop"])
print("Model Answer:", ans_3)
print("Ground Truth:", sample_b["target_3hop"])

=== ZERO-SHOT SAMPLE TEST (TASK B: MULTI-HOP) ===
Context:
 Fact 1: Lisinopril TARGETS BCR_ABL_Kinase.
Fact 2: BCR_ABL_Kinase is ENCODED_BY gene PDCD1.
Fact 3: Gene PDCD1 is ASSOCIATED_WITH Type2Diabetes.
Query (2-Hop): Based on the pathway facts, which Gene is modulated when Lisinopril targets its primary protein?
Model Answer: Gene PDCD1
Ground Truth: PDCD1
----------------------------------------
Query (3-Hop): Following the 3-hop biological chain from Lisinopril, which Disease is causally linked to this target pathway?
Model Answer: Type2Diabetes
Ground Truth: Type2Diabetes


### Step 6: Evaluate Task C — Zero-Shot Link Prediction

In [11]:
def predict_link_zero_shot(head, tail, prompt_text):
    prompt_messages = [
        {
            "role": "system",
            "content": "Predict the single valid ontology relation connecting Head to Tail. Reply with ONLY the relation name."
        },
        {
            "role": "user",
            "content": prompt_text
        }
    ]
    formatted = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=25, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

print("=== ZERO-SHOT SAMPLE TEST (TASK C: LINK PREDICTION) ===")
sample_c = task_c_data[0]
pred_rel = predict_link_zero_shot(sample_c["head"], sample_c["tail"], sample_c["prompt"])
print(f"Head: {sample_c['head']} | Tail: {sample_c['tail']}")
print("Predicted Relation:", pred_rel)
print("Target Relation   :", sample_c["target_relation"])

=== ZERO-SHOT SAMPLE TEST (TASK C: LINK PREDICTION) ===
Head: Lisinopril | Tail: ImmuneColitis
Predicted Relation: CAUSES_SIDE_EFFECT
Target Relation   : CAUSES_SIDE_EFFECT


---
## 📊 Zero-Training Baseline Summary & Key Questions

When you run this benchmark in Colab, you will immediately observe three baseline phenomena:
1. **Formatting Inconsistency in Task A**: Untuned base models output conversational pleasantries, markdown fences, or rename schema predicates (e.g. `INDICATED_FOR` instead of the formal ontology relation `TREATS`).
2. **Hop Degradation in Task B**: Models succeed reliably at 1-hop, but as reasoning hops increase to 2-hop and 3-hop, hallucination increases sharply.
3. **Link Prediction Ambiguity in Task C**: Without explicit ontology fine-tuning, the model may invent free-form relations outside the allowed vocabulary.

**This established baseline provides the exact numeric hurdle that Track 1 (Pure LLM Fine-Tuning) and Track 2 (GNN + LLM) must beat!**